In [1]:
import csv
import gzip
import re

#input and output files for SNPS

vep_file = "M4_family_singletons-VeP.txt"
vcf_file = "M4_family_singletons.vcf.gz"
output_file = "FINAL_M4_candidate_mutations_summary.csv"

#exact parental sample names
#names must match the VCF sample columns exactly
parentals = {
    "MN1806CN-1_founder-prep04_Rep1-MN1806CN-1_founder-prep04_Rep1-CTTAAGCTTAAG",
    "MN1806CN-1_founder-prep04_Rep2-MN1806CN-1_founder-prep04_Rep2-TGGCTTTGGCTT",
    "MN1806CN-1_founder-prep11_Rep1-MN1806CN-1_founder-prep11_Rep1-GAGTGCGAGTGC",
    "MN1806CN-1_founder-prep11_Rep2-MN1806CN-1_founder-prep11_Rep2-ATGGTGATGGTG",
}

MIN_DP = 10
MAX_DP = 100
MIN_GQ = 20
#match what was done with R filtering

#

def parse_uploaded_variation(uploaded):
    """
    Parse VEP IDs such as:
        Gm01_12345_A/G
        1_12345_A/G

    Returns exact (chromosome, position, REF, ALT)
    """
    match = re.match(r"^(.+)_([0-9]+)_([^/]+)/([^/]+)$", uploaded)

    if match is None:
        return None

    chrom, pos, ref, alt = match.groups()
    return chrom, str(pos), ref, alt


def classify_ad(ad, gq):
    """
    Apply the same AD-derived DP, GQ, and AB rules as in R filtering
    """
    if ad in (None, "", ".", "./.") or gq in (None, "", "."):
        return None, None, None, None

    parts = ad.split(",")

    #SNPs must be biallelic.
    if len(parts) != 2:
        return None, None, None, None

    try:
        ref_reads = int(parts[0])
        alt_reads = int(parts[1])
        gq_value = float(gq)
    except (ValueError, TypeError):
        return None, None, None, None

    dp = ref_reads + alt_reads

    if dp < MIN_DP or dp > MAX_DP or gq_value < MIN_GQ:
        return None, ref_reads, alt_reads, None

    if dp == 0:
        return None, ref_reads, alt_reads, None

    ab = alt_reads / dp

    if ab < 0.4:
        call = "REF"
    elif ab < 0.9:
        call = "HET"
    else:
        call = "HOM_ALT"

    return ab, ref_reads, alt_reads, call


def get_family_id(sample):
    """Extract the MEE family identifier from a mutant sample name."""
    match = re.match(r"^(MEE\d+)", sample)

    if match is None:
        raise ValueError(
            f"Cannot determine MEE family from sample name: {sample}"
        )

    return match.group(1)


#read VEP annotations and keep only the high and moderate impact only.
# ============================================================

variants = {}

with open(vep_file, "r") as f:
    for line in f:
        if line.startswith("#"):
            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) < 14:
            continue

        uploaded = fields[0]
        gene = fields[3]
        consequence = fields[6]
        protein_position = fields[9]
        amino_acids = fields[10]
        codons = fields[11]
        extra = fields[13]

        if "IMPACT=HIGH" in extra:
            impact = "HIGH"
        elif "IMPACT=MODERATE" in extra:
            impact = "MODERATE"
        else:
            continue

        key = parse_uploaded_variation(uploaded)

        if key is None:
            continue

        symbol_match = re.search(r"(?:^|;)SYMBOL=([^;]+)", extra)
        symbol = symbol_match.group(1) if symbol_match else ""

        annotation = {
            "gene": gene,
            "symbol": symbol,
            "consequence": consequence,
            "impact": impact,
            "protein_position": protein_position,
            "amino_acids": amino_acids,
            "codons": codons,
        }

        #keep one annotation per exact SNP key, so 
        #one-row-per-variant format
        variants[key] = annotation

print("VEP HIGH/MODERATE SNP annotations loaded:", len(variants))



#read the vcf and summarize the SNP carriers

with gzip.open(vcf_file, "rt") as vcf, \
     open(output_file, "w", newline="") as out:

    writer = csv.writer(out)

    writer.writerow([
        "Gene_ID",
        "Gene_symbol",
        "Chromosome",
        "Position",
        "REF",
        "ALT",
        "Consequence",
        "Impact",
        "Protein_position",
        "AA_change",
        "Codon_change",
        "Num_carriers",
        "Num_families",
        "HET_samples",
        "HOM_ALT_samples",
        "Carrier_details",
    ])

    samples = []
    parental_indices = []
    mutant_indices = []

    n_biallelic_snps = 0
    n_vep_matches = 0
    n_unmatched_vep = 0
    n_parental_exclusions = 0
    n_multifamily_exclusions = 0
    n_written = 0

    for line in vcf:

        if line.startswith("##"):
            continue

        if line.startswith("#CHROM"):
            header = line.rstrip("\n").split("\t")
            samples = header[9:]
            sample_index = {
                sample: i for i, sample in enumerate(samples)
            }

            found_parentals = parentals.intersection(samples)
            missing_parentals = parentals - set(samples)

            print("Samples in VCF:", len(samples))
            print("Parental samples found:", len(found_parentals))

            if missing_parentals:
                raise ValueError(
                    "Parental samples missing from VCF:\n"
                    + "\n".join(sorted(missing_parentals))
                )

            parental_indices = [
                sample_index[s] for s in parentals
            ]

            mutant_indices = [
                i for i, sample in enumerate(samples)
                if sample not in parentals
            ]

            continue

        fields = line.rstrip("\n").split("\t")

        if len(fields) < 10:
            continue

        chrom, pos, ref, alt = (
            fields[0], fields[1], fields[3], fields[4]
        )

        #exclude multiallelic records if any exist
        if "," in alt:
            continue

        #keep SNPs only: one base REF and one base ALT
        if len(ref) != 1 or len(alt) != 1:
            continue

        n_biallelic_snps += 1

        #exact SNP
        key = (chrom, str(pos), ref, alt)
        annotation = variants.get(key)

        if annotation is None:
            n_unmatched_vep += 1
            continue

        n_vep_matches += 1

        format_fields = fields[8].split(":")

        if "AD" not in format_fields or "GQ" not in format_fields:
            continue

        ad_index = format_fields.index("AD")
        gq_index = format_fields.index("GQ")
        gt_index = (
            format_fields.index("GT")
            if "GT" in format_fields else None
        )

        calls = {}

        for i, sample in enumerate(samples):
            sample_values = fields[9 + i].split(":")

            ad = (
                sample_values[ad_index]
                if ad_index < len(sample_values) else "."
            )
            gq = (
                sample_values[gq_index]
                if gq_index < len(sample_values) else "."
            )
            gt = (
                sample_values[gt_index]
                if gt_index is not None
                and gt_index < len(sample_values)
                else "."
            )

            ab, ref_reads, alt_reads, call = classify_ad(ad, gq)

            calls[sample] = {
                "ab": ab,
                "ref_reads": ref_reads,
                "alt_reads": alt_reads,
                "call": call,
                "gt": gt,
            }

        #exclude sites with confident parental ALT evidence
        parental_alt_samples = [
            samples[i]
            for i in parental_indices
            if calls[samples[i]]["call"] in ("HET", "HOM_ALT")
        ]

        if parental_alt_samples:
            n_parental_exclusions += 1
            continue

        #collect ALT carriers by family.
        family_carriers = {}

        for i in mutant_indices:
            sample = samples[i]
            info = calls[sample]

            if info["call"] not in ("HET", "HOM_ALT"):
                continue

            family_id = get_family_id(sample)
            family_carriers.setdefault(family_id, []).append(
                (sample, info)
            )

        #exactly one mutant family must carry ALT
        #multiple sibling carriers within that family are allowed
        if len(family_carriers) != 1:
            if len(family_carriers) > 1:
                n_multifamily_exclusions += 1
            continue

        het_samples = []
        hom_samples = []
        carrier_details = []

        for family_samples in family_carriers.values():
            for sample, info in family_samples:

                if info["call"] == "HET":
                    het_samples.append(sample)
                else:
                    hom_samples.append(sample)

                carrier_details.append(
                    f"{sample}(GT={info['gt']},"
                    f"AD={info['ref_reads']},{info['alt_reads']},"
                    f"AB={info['ab']:.3f},{info['call']})"
                )

        carriers = het_samples + hom_samples

        if not carriers:
            continue

        writer.writerow([
            annotation["gene"],
            annotation["symbol"],
            chrom,
            pos,
            ref,
            alt,
            annotation["consequence"],
            annotation["impact"],
            annotation["protein_position"],
            annotation["amino_acids"],
            annotation["codons"],
            len(carriers),
            len(family_carriers),
            ";".join(het_samples),
            ";".join(hom_samples),
            "; ".join(carrier_details),
        ])

        n_written += 1

print()
print("Finished.")
print("Biallelic SNPs examined:", n_biallelic_snps)
print("SNPs matching HIGH/MODERATE VEP annotations:", n_vep_matches)
print("SNPs unmatched to VEP annotations:", n_unmatched_vep)
print("Sites excluded due to parental ALT evidence:", n_parental_exclusions)
print("Sites excluded due to multiple mutant families:", n_multifamily_exclusions)
print("Sites written:", n_written)
print("Output:", output_file)

VEP HIGH/MODERATE SNP annotations loaded: 32686
Samples in VCF: 175
Parental samples found: 4

Finished.
Biallelic SNPs examined: 717141
SNPs matching HIGH/MODERATE VEP annotations: 32686
SNPs unmatched to VEP annotations: 684455
Sites excluded due to parental ALT evidence: 0
Sites excluded due to multiple mutant families: 0
Sites written: 32686
Output: FINAL_M4_candidate_mutations_summary.csv
